Imports & Setups

In [1]:
from termcolor import colored
from langgraph_utils import console
from langgraph.store.memory import InMemoryStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings


Store

In [2]:
ltm_store = InMemoryStore()


Namespaces

In [3]:
namespace_buh = ("user", "buh")
namespace_pattagobi = ("user", "pattagobi")

Memories

In [4]:
memories_buh = [
    ("food_preference", "Favorite food: Garlic Guajillo Shrimps."),
    ("profession", "Profession: Professional gaslighter."),
    ("company", "Employer: Lumon Industries."),
    ("season_preference", "Favorite season: Winter."),
]

memories_pattagobi = [
    ("food_preference", "Favorite food: Sun."),
    ("profession", "Profession: Producer."),
    ("company", "Employer: Touch Grass.Inc."),
    ("season_preference", "Favorite season: Spices."),
]


put ops

In [5]:
for memory in memories_buh:
    ltm_store.put(
        namespace=namespace_buh,
        key=memory[0],
        value={
            "text": memory[1],
        },
    )


In [6]:
for memory in memories_pattagobi:
    ltm_store.put(
        namespace=namespace_pattagobi,
        key=memory[0],
        value={
            "text": memory[1],
        },
    )


get ops

In [7]:
console.print_json(data=ltm_store.get(namespace=namespace_buh, key="food_preference").dict())

{
  "namespace": [
    "user",
    "buh"
  ],
  "key": "food_preference",
  "value": {
    "text": "Favorite food: Garlic Guajillo Shrimps."
  },
  "created_at": "2026-07-08T16:59:26.289715+00:00",
  "updated_at": "2026-07-08T16:59:26.289715+00:00"
}


In [8]:
console.print_json(data=ltm_store.get(namespace=namespace_pattagobi, key="company").dict())

{
  "namespace": [
    "user",
    "pattagobi"
  ],
  "key": "company",
  "value": {
    "text": "Employer: Touch Grass.Inc."
  },
  "created_at": "2026-07-08T16:59:26.292383+00:00",
  "updated_at": "2026-07-08T16:59:26.292383+00:00"
}


search ops

In [9]:
items = ltm_store.search(namespace_buh)
console.print_json(data={item.key: item.value["text"] for item in items})


{
  "food_preference": "Favorite food: Garlic Guajillo Shrimps.",
  "profession": "Profession: Professional gaslighter.",
  "company": "Employer: Lumon Industries.",
  "season_preference": "Favorite season: Winter."
}


In [10]:
items = ltm_store.search(namespace_pattagobi)
console.print_json(data={item.key: item.value["text"] for item in items})


{
  "food_preference": "Favorite food: Sun.",
  "profession": "Profession: Producer.",
  "company": "Employer: Touch Grass.Inc.",
  "season_preference": "Favorite season: Spices."
}


### Semantic Search

Embeddings

In [11]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

Store

In [12]:
ltm_store_embedded = InMemoryStore(
    index={
        "embed": embeddings,
        "dims": len(embeddings.embed_query("hello")),
        "fields": ["text"],
    }
)

put ops

In [13]:
for memory in memories_buh:
    ltm_store_embedded.put(
        namespace=namespace_buh,
        key=memory[0],
        value={
            "text": memory[1],
        },
    )


search (semantic) ops

In [14]:
items = {
    item.key: item.value["text"]
    for item in ltm_store_embedded.search(
        namespace_buh,
        query="where does user work?",
        limit=1,
    )
}

console.print_json(data=items)


{
  "company": "Employer: Lumon Industries."
}


In [15]:
items = {
    item.key: item.value["text"]
    for item in ltm_store_embedded.search(
        namespace_buh,
        query="what are user's preferences",
        limit=2,
    )
}
console.print_json(data=items)

{
  "season_preference": "Favorite season: Winter.",
  "food_preference": "Favorite food: Garlic Guajillo Shrimps."
}
